In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType

MAPEAMENTO_COOPERATIVAS = {
    "DEPOSITO_A_VISTA": ["41100000", "4110000006"],
    "DEPOSITO_A_PRAZO": ["41500000","41500002","4150000004","43200001","4320000009","43200000","41400009","4140000007","41400000"],
    "PATRIMONIO_LIQUIDO": ["60000002","60000000","6000000004"],
    "CREDITO_COMERCIAL": ["16100004","1612000002","16200007","1620000001","16100000","16200000"],
    "REPASSES + BNDES + FUNDOS": ["16300000","1630000008"],
    "ATIVO": ["39999993","39999990","3999999009"],
    "COMPENSACAO": ["30000000","30000001","3000000007"],
    "RESULTADO": ["70000009","7000000003","80000006","8000000002","70000000","80000000"]
}

MAPEAMENTO_BANCOS = {
    "DEPOSITO_A_VISTA": ["VERBETE_401_SERVICOS_PUBLICOS", "VERBETE_402_ATIVIDADES_EMPRESARIAIS", "VERBETE_403_ESPECIAIS_DO_TESOURO_NACIONAL", "VERBETE_404_SALDOS_CREDORES_EM_CONTAS_DE_EMPRESTIMOS_E_FINAN", "VERBETE_411_DE_PESSOAS_FISICAS", "VERBETE_412_DE_PESSOAS_JURIDICAS", "VERBETE_413_DE_INSTITUICOES_FINANCEIRAS", "VERBETE_414_JUDICIAIS", "VERBETE_415_OBRIGATORIOS", "VERBETE_416_PARA_INVESTIMENTOS", "VERBETE_417_VINCULADOS", "VERBETE_418_DEMAIS_DEPOSITOS", "VERBETE_419_SLD_CRED_CTAS_EMPR_FINANC_OUTR"],
    "DEPOSITO_A_PRAZO": ["VERBETE_432_DEPOSITOS_A_PRAZO"],
    "CREDITO_COMERCIAL": ["VERBETE_161_EMPRES_E_TIT_DESCONTADOS"],
    "REPASSES + BNDES + FUNDOS": ["VERBETE_162_FINANCIAMENTOS", "VERBETE_163_FIN_RURAIS_AGRICUL_CUST/INVEST"],
    "RESULTADO": ["VERBETE_710_CONTAS_DE_RESULTADO"],
    "POUPANCA": ["VERBETE_420_DEPOSITOS_DE_POUPANCA"]
}

def limpar_numero(col_name):
    return F.coalesce(
        F.regexp_replace(F.regexp_replace(F.col(col_name), "\\.", ""), ",", ".").cast(DoubleType()),
        F.lit(0.0)
    )

df_coop_bronze = spark.table("bronze_coop")

expressao_mapa_coop = F.when(F.lit(False), F.lit(None))
for ind, contas in MAPEAMENTO_COOPERATIVAS.items():
    expressao_mapa_coop = expressao_mapa_coop.when(F.col("CONTA").isin(contas), F.lit(ind))

df_coop_silver = (
    df_coop_bronze
    .filter(F.trim(F.col("DOCUMENTO")) == "4010")
    .withColumn("indicador", expressao_mapa_coop)
    .filter(F.col("indicador").isNotNull())
    .withColumn("valor", limpar_numero("SALDO"))
    .withColumn("datamovimento", F.to_date(F.concat(F.col("#DATA_BASE"), F.lit("01")), "yyyyMMdd"))
    .select("datamovimento", "CNPJ", "indicador", "valor")
)

df_coop_silver.write.format("delta").mode("overwrite").saveAsTable("silver_coop")
df_estban_bronze = spark.table("bronze_estban")
df_estban = df_estban_bronze.withColumn(
    "datamovimento", F.to_date(F.concat(F.col("#DATA_BASE"), F.lit("01")), "yyyyMMdd")
)

colunas_df = df_estban.columns
for ind, cols_lista in MAPEAMENTO_BANCOS.items():
    cols_validas = [c for c in cols_lista if c in colunas_df]
    if cols_validas:
        colunas_limpas = [limpar_numero(c) for c in cols_validas]
        df_estban = df_estban.withColumn(ind, sum(colunas_limpas))
    else:
        df_estban = df_estban.withColumn(ind, F.lit(0.0))

colunas_id = [c for c in ["datamovimento", "UF", "CODMUN", "MUNICIPIO", "CNPJ", "NOME_INSTITUICAO", "NUMEROS_AGENCIA"] if c in colunas_df]
inds_para_derreter = list(MAPEAMENTO_BANCOS.keys())

expr_unpivot = f"stack({len(inds_para_derreter)}, " + ", ".join([f"'{k}', `{k}`" for k in inds_para_derreter]) + ") as (indicador, valor)"

df_estban_silver = (
    df_estban
    .select(*colunas_id, F.expr(expr_unpivot))
)

df_estban_silver.write.format("delta").mode("overwrite").saveAsTable("silver_estban")